# Computing ED2 and EDGE2 scores

This notebook demonstrates how to iterate through a distribution of trees, compute ED2 and EDGE2 scores, and write out the distributions of scores.

Note that the full output file has 2.3 million rows (one per species), so it is too large to open in e.g. Excel. The final function below can produce an additional file with the list cropped down to e.g. the top 10 species per family, which is more manageable.

We will build up gradually to the full computation. We will begin by using a small tree of birds; then try the full tree. Then we will loop over the birds tree, and finally loop over the full tree - which can take a while, so we want to be sure everything works first.

## Loading a tree

First, check you can import the necessary Python modules.

In [1]:
import numpy as np

import tree_loading
import tree_labelling
import tree_dating
import tree_metrics

We are going to load a tree and annotate it with the taxonomy. By default we will look for the taxonomy file `ott3.7.3/taxonomy.tsv`.

We will load a tree from the distribution of trees available at .... Here we put the trees in a folder `trees/`, and we will load tree 407, which is the tree in the topological variation distribution that has the median PD.

In [2]:
# Load taxonomic info
taxa = tree_loading.load_metadata(date_cache=None, annotations=None)

In [13]:
# Load and annotate tree number 407, the "median" tree
tree = tree_loading.build_and_annotate_tree(phylogeny_nodes=None, 
                                            taxa=taxa,
                                            tree_filename="trees/dated_tree_topo_sample_407.tre",
                                            has_branch_lengths=True,
                                            suppress_logging=True)

In [ ]:
tree_labelling.add_anc_ranks(tree)

## Computing EDGE2

Now we assign IUCN Red List status to each species, where available. By default we use `config/latest_iucn_2025.csv` (provided by Jialiang Guo).

In [4]:
tree_metrics.assign_iucn_status(tree)

Then we convert statuses to probabilities using the curve from the EDGE2 protocol (Gumbs et al. 2024). We sample from the relevant part of the curve for each status; to use the median instead, set `randomise_risk=False`. Where no status is available we randomly sample an extinction probability from the curve.
![Status -> extinction probability curve](config/iucn_curve.png)

In [5]:
rng = np.random.default_rng(seed=1)
tree_metrics.assign_extinction_risks(tree, rng=rng, randomise_risk=True)

Finally, we compute ED2 and EDGE2 scores. This will annotate all the leaves of the tree with the properties `ed2_score` and `edge2_score`.

In [10]:
tree_metrics.compute_edge2_scores(tree)
for n in tree.search_nodes(name="Homo_sapiens_ott770315"):
    print(n.name, n.props["ed2_score"], n.props["edge2_score"])
    break

Homo_sapiens_ott770315 8.976553727819056 0.35397063886178953


## Computing a distribution of scores

Now we have got the basic code working, we can produce a distribution of scores. First we will loop over a single birds tree. (The extinction probabilities are sampled from a curve, so we will still get a distribution of scores even with only one tree.)

In [ ]:
import importlib
importlib.reload(tree_metrics)

import datetime
rng = np.random.default_rng(seed=1)

scores_dict = {}
num_itrs = 10

start_time = datetime.datetime.now()
for itr in range(num_itrs):
    print("Tree number",
          itr+1,
          "/ projected end time:",
          start_time + num_itrs*(datetime.datetime.now() - start_time)/itr if itr > 0 else "first iteration, no estimate yet")
    
    tree = tree_loading.build_and_annotate_tree(phylogeny_nodes=None, 
                                                taxa=taxa,
                                                tree_filename="config/birds.tre",
                                                has_branch_lengths=True,
                                                suppress_logging=True)
    tree_labelling.add_anc_ranks(tree)

    tree_metrics.assign_iucn_status(tree)
    tree_metrics.assign_extinction_risks(tree, rng=rng, randomise_risk=True)
    tree_metrics.compute_edge2_scores(tree, scores_dict=scores_dict)

print("Now writing out scores")
tree_metrics.write_edge2_scores("edge2_scores_birds.csv", scores_dict)
print("Done.")

## A distribution of EDGE2 scores for all extant described species

Finally, let's try looping over some complete trees. This time we will limit the output file to the top 100 species per phylum, by mean EDGE2 score. Let's start with 10 trees; but this can be modified to run through the full distribution of trees if desired.

In [ ]:
import importlib
importlib.reload(tree_metrics)

import datetime
rng = np.random.default_rng(seed=1)

scores_dict = {}
num_itrs = 10

start_time = datetime.datetime.now()
for itr in range(num_itrs):
    print("Tree number",
          itr+1,
          "/ projected end time:",
          start_time + num_itrs*(datetime.datetime.now() - start_time)/itr if itr > 0 else "first iteration, no estimate yet")
    
    tree = tree_loading.build_and_annotate_tree(phylogeny_nodes=None, 
                                                taxa=taxa,
                                                tree_filename="trees/dated_tree_topo_sample_{:d}.tre".format(itr+1),
                                                has_branch_lengths=True,
                                                suppress_logging=True)
    tree_labelling.add_anc_ranks(tree)

    tree_metrics.assign_iucn_status(tree)
    tree_metrics.assign_extinction_risks(tree, rng=rng, randomise_risk=True)
    tree_metrics.compute_edge2_scores(tree, scores_dict=scores_dict)

print("Now writing out scores")
tree_metrics.write_edge2_scores("edge2_scores_all.csv", scores_dict, per_phylum=100)

Tree number 1 / projected end time: first iteration, no estimate yet
Tree number 2 / projected end time: 2026-02-12 13:26:30.951880
Tree number 3 / projected end time: 2026-02-12 13:23:55.954535
